# Fundamental vertical: self-contained end-to-end notebook

**Author:** Jan
**Last updated:** 2026-05-29

End-to-end version of the fundamental vertical, matching the structure used by
the other three verticals (Sacha, Rayane, Cesar) so the orchestrator can treat
all four uniformly. Strategy logic is identical to `src/fundamental.py`: a
GLD tilt driven by the smoothed monthly change in the 10-year real yield
(FRED `DFII10`), z-scored over 24 months and capped at +/- 15pp from the
25% baseline. The remaining three legs (ACWI, AGG, BSV) split (1 - GLD)
equally. Canonical setup, KPI bank, turnover-cap projector, and submission
writer are inlined verbatim from `shared/canonical.py`.


## 1. Setup

In [1]:
from __future__ import annotations

import json
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
import yfinance as yf

notebook_dir = Path.cwd()
if notebook_dir.name == "notebooks":
    VERTICAL_ROOT = notebook_dir.parent
else:
    VERTICAL_ROOT = notebook_dir
REPO_ROOT = VERTICAL_ROOT.parent
DATA_DIR = REPO_ROOT / "data"
SUBMISSIONS_DIR = VERTICAL_ROOT / "submissions"
RESULTS_DIR = VERTICAL_ROOT / "results"
SUBMISSIONS_DIR.mkdir(exist_ok=True)
RESULTS_DIR.mkdir(exist_ok=True)
print(f"REPO_ROOT:        {REPO_ROOT}")
print(f"DATA_DIR:         {DATA_DIR}  (exists={DATA_DIR.exists()})")
print(f"SUBMISSIONS_DIR:  {SUBMISSIONS_DIR}")


REPO_ROOT:        c:\Users\sacha\Downloads\algo-trading-group
DATA_DIR:         c:\Users\sacha\Downloads\algo-trading-group\data  (exists=True)
SUBMISSIONS_DIR:  c:\Users\sacha\Downloads\algo-trading-group\jan-fundamental\submissions


In [2]:
# === BEGIN canonical:constants ===
UNIVERSE = ["ACWI", "AGG", "GLD", "BSV"]
TEAM_ID = "TeamXX"
TRAIN_END = pd.Timestamp("2024-12-31")
VAL_END = pd.Timestamp("2025-12-31")
TURNOVER_CAP = 0.25
PERIODS_PER_YEAR = 52
RISK_FREE_RATE = 0.0
MAR = 0.0
MIN_PERIODS_FOR_RATIO = 24
RNG_SEED = 42

np.random.seed(RNG_SEED)
pd.set_option("display.float_format", "{:.4f}".format)
# === END canonical:constants ===

## 2. Canonical KPI / turnover / submission blocks (inlined)

In [3]:
# === BEGIN canonical:kpis ===
def compute_kpis(
    returns: pd.Series,
    benchmark: pd.Series | None = None,
    *,
    periods_per_year: int = PERIODS_PER_YEAR,
    risk_free_rate: float = RISK_FREE_RATE,
    mar: float = MAR,
    min_periods_for_ratio: int = MIN_PERIODS_FOR_RATIO,
) -> dict:
    """Weekly KPI bank shared by every vertical.

    Sharpe uses mean excess return over sample-std (ddof=1), annualised by
    sqrt(periods_per_year). Sortino uses the textbook downside deviation:
    sqrt(mean(min(excess - mar_per_period, 0)^2)). Returns NaN for the
    ratio metrics when the sample is shorter than min_periods_for_ratio.
    """
    r = returns.dropna()
    if r.empty:
        return {}
    rf_per = risk_free_rate / periods_per_year
    mar_per = mar / periods_per_year
    ann = float(np.sqrt(periods_per_year))

    excess = r - rf_per
    mean_excess = float(excess.mean())
    vol = float(excess.std(ddof=1))

    downside = np.minimum(excess - mar_per, 0.0)
    downside_dev = float(np.sqrt(np.mean(np.square(downside))))

    equity = (1.0 + r).cumprod()
    drawdown = equity / equity.cummax() - 1.0
    n = len(r)
    n_years = n / periods_per_year
    total_return = float(equity.iloc[-1] - 1.0)
    cagr = float(equity.iloc[-1] ** (1.0 / n_years) - 1.0) if n_years > 0 else float("nan")
    max_dd = float(drawdown.min())

    enough = n >= min_periods_for_ratio
    sharpe = (mean_excess / vol) * ann if enough and vol > 0 else float("nan")
    sortino = (mean_excess / downside_dev) * ann if enough and downside_dev > 0 else float("nan")
    calmar = cagr / abs(max_dd) if max_dd < 0 and not np.isnan(cagr) else float("nan")

    out = {
        "n_periods": int(n),
        "total_return": total_return,
        "CAGR": cagr,
        "ann_return": float(r.mean() * periods_per_year),
        "ann_vol": float(r.std(ddof=1) * ann),
        "Sharpe": float(sharpe),
        "Sortino": float(sortino),
        "max_drawdown": max_dd,
        "calmar": float(calmar),
        "hit_rate": float((r > 0).mean()),
        "best_period": float(r.max()),
        "worst_period": float(r.min()),
    }
    if benchmark is not None:
        bench = benchmark.reindex(r.index).dropna()
        active = (r - bench).dropna()
        if len(active) >= min_periods_for_ratio and active.std(ddof=1) > 0:
            out["active_sharpe_vs_bench"] = float(
                (active.mean() / active.std(ddof=1)) * ann
            )
        else:
            out["active_sharpe_vs_bench"] = float("nan")
        out["excess_ann_return"] = (
            float(active.mean() * periods_per_year) if not active.empty else float("nan")
        )
    return out
# === END canonical:kpis ===

In [4]:
# === BEGIN canonical:turnover ===
def project_to_turnover_ball(
    target: pd.Series,
    previous: pd.Series,
    *,
    cap: float = TURNOVER_CAP,
) -> pd.Series:
    """L1-ball projection of target toward previous so sum(|target-previous|) <= cap.

    Returns a Series indexed by UNIVERSE, clipped to [0,1] and renormalised to
    sum to 1. Matches the math each vertical already used independently.
    """
    target = target.reindex(UNIVERSE).astype(float)
    previous = previous.reindex(UNIVERSE).astype(float)
    delta = target - previous
    total = float(np.abs(delta).sum())
    if total <= cap + 1e-12:
        out = target.copy()
    else:
        out = previous + delta * (cap / total)
    out = out.clip(lower=0.0, upper=1.0)
    s = float(out.sum())
    return out / s if s > 0 else target
# === END canonical:turnover ===

In [5]:
# === BEGIN canonical:submission ===
def write_submission_csv(
    weights_fraction: pd.Series,
    week: "pd.Timestamp | str",
    team_id: str,
    out_dir: Path,
    *,
    vertical_tag: str | None = None,
    previous_path: Path | None = None,
    enforce_turnover: bool = True,
) -> Path:
    """Write the weekly submission CSV in the format mandated by CLAUDE.md.

    Input weights are fractions in [0,1] summing to 1. They are scaled to
    percentages, rounded to two decimals, and the residual is added to the
    largest weight so the row sums to exactly 100.00. Per-vertical draft files
    get a {team_id}_{vertical_tag}_{date}.csv name; the final team submission
    uses {team_id}_{date}.csv. Header is week,team_id,acwi,agg,gld,bsv.
    """
    week_ts = pd.Timestamp(week)
    week_str = week_ts.strftime("%Y-%m-%d")
    w = weights_fraction.reindex(UNIVERSE).astype(float) * 100.0
    w = w.round(2)
    residual = round(100.0 - float(w.sum()), 2)
    largest = w.idxmax()
    w.loc[largest] = round(float(w.loc[largest]) + residual, 2)
    if not ((w >= 0).all() and (w <= 100).all()):
        raise ValueError(f"weights out of [0,100]: {w.to_dict()}")
    if abs(float(w.sum()) - 100.0) > 1e-6:
        raise ValueError(f"weights do not sum to 100: {float(w.sum())}")
    if enforce_turnover and previous_path is not None and Path(previous_path).exists():
        prev_df = pd.read_csv(previous_path)
        prev_w = pd.Series(
            {k: float(prev_df.iloc[0][k.lower()]) for k in UNIVERSE},
        ).reindex(UNIVERSE)
        turnover_pp = float((w - prev_w).abs().sum())
        if turnover_pp - 25.0 > 1e-9:
            raise ValueError(
                f"turnover {turnover_pp:.2f}pp > 25.00pp vs {Path(previous_path).name}"
            )
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    name = (
        f"{team_id}_{vertical_tag}_{week_str}.csv"
        if vertical_tag
        else f"{team_id}_{week_str}.csv"
    )
    path = out_dir / name
    row = {
        "week": week_str,
        "team_id": team_id,
        "acwi": w["ACWI"],
        "agg": w["AGG"],
        "gld": w["GLD"],
        "bsv": w["BSV"],
    }
    pd.DataFrame([row], columns=["week", "team_id", "acwi", "agg", "gld", "bsv"]).to_csv(
        path, index=False, float_format="%.2f",
    )
    return path
# === END canonical:submission ===

## 3. Data loading (FRED + yfinance, cached on disk)

In [6]:
def _load_env() -> None:
    env_path = REPO_ROOT / ".env"
    if not env_path.exists():
        return
    for line in env_path.read_text().splitlines():
        line = line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, _, value = line.partition("=")
        os.environ.setdefault(key.strip(), value.strip())


def _fred_api_key() -> str:
    _load_env()
    key = os.environ.get("FRED_API_KEY")
    if not key:
        raise RuntimeError("FRED_API_KEY not set. Copy .env.example to .env and add your key.")
    return key


FRED_BASE = "https://api.stlouisfed.org/fred"


def load_fred_series(series_id: str, start: str = "1990-01-01", refresh: bool = False) -> pd.Series:
    DATA_DIR.mkdir(exist_ok=True)
    cache = DATA_DIR / f"{series_id}.csv"
    if cache.exists() and not refresh:
        cached = pd.read_csv(cache, index_col="date", parse_dates=["date"])[series_id]
        return cached.loc[start:]
    params = {"series_id": series_id, "api_key": _fred_api_key(), "file_type": "json"}
    response = requests.get(f"{FRED_BASE}/series/observations", params=params, timeout=30)
    response.raise_for_status()
    observations = response.json()["observations"]
    df = pd.DataFrame(observations)
    df["date"] = pd.to_datetime(df["date"])
    df["value"] = pd.to_numeric(df["value"], errors="coerce")
    df = df.set_index("date")[["value"]].rename(columns={"value": series_id})
    df.to_csv(cache)
    return df[series_id].loc[start:]


def load_fred_as_of(series_id: str, as_of_date, start: str = "1990-01-01") -> pd.Series:
    """Series as of `as_of_date`. For daily yields with no meaningful revisions,
    falls back to filtering by observation date."""
    as_of_ts = pd.Timestamp(as_of_date)
    series = load_fred_series(series_id, start=start)
    return series[series.index <= as_of_ts]


def load_etf_prices(tickers, start: str = "2003-01-01", refresh: bool = False) -> pd.DataFrame:
    if isinstance(tickers, str):
        tickers = [tickers]
    DATA_DIR.mkdir(exist_ok=True)
    prices = {}
    for ticker in tickers:
        cache = DATA_DIR / f"{ticker}_prices.csv"
        if cache.exists() and not refresh:
            prices[ticker] = pd.read_csv(cache, index_col=0, parse_dates=True).iloc[:, 0]
            continue
        raw = yf.download(ticker, period="max", auto_adjust=True, progress=False)
        if raw.empty:
            raise RuntimeError(f"No data returned from yfinance for {ticker}")
        if isinstance(raw.columns, pd.MultiIndex):
            raw.columns = raw.columns.get_level_values(0)
        close = raw["Close"].rename(ticker)
        close.to_csv(cache, header=True)
        prices[ticker] = close
    return pd.DataFrame(prices).loc[start:]


## 4. Signal logic: GLD tilt from smoothed Δ DFII10

Identical to `src/fundamental.py`. Real yields drive gold; when smoothed
month-on-month changes in DFII10 are negative (real yields falling), the model
tilts toward GLD. The tilt is `MAX_TILT * tanh(-z)` where `z` is the z-score
of the smoothed delta over a 24-month window. Other three legs split (1 - GLD)
equally.

In [7]:
SMOOTH_WINDOW = 3
LOOKBACK = 24
BASELINE_WEIGHT = 0.25
MAX_TILT = 0.15


def gld_tilt(as_of_date) -> float:
    as_of_ts = pd.Timestamp(as_of_date)
    series = load_fred_as_of("DFII10", as_of_ts, start="2000-01-01")
    monthly = series.resample("ME").last()
    monthly = monthly.loc[monthly.index <= as_of_ts]
    if monthly.empty:
        raise ValueError(f"No completed DFII10 month available on or before {as_of_ts.date()}")
    delta_smoothed = monthly.diff().rolling(SMOOTH_WINDOW).mean()
    z = (delta_smoothed - delta_smoothed.rolling(LOOKBACK).mean()) / delta_smoothed.rolling(LOOKBACK).std()
    tilt = MAX_TILT * np.tanh(-z)
    valid_tilt = tilt.dropna()
    if valid_tilt.empty:
        raise ValueError(
            f"Insufficient DFII10 history on {as_of_ts.date()} for a {LOOKBACK}-month z-score"
        )
    return float(valid_tilt.iloc[-1])


def get_weights(as_of_date) -> pd.Series:
    """Canonical signal contract: Series indexed UNIVERSE, in [0,1], summing to 1."""
    tilt = gld_tilt(pd.Timestamp(as_of_date))
    gld = float(np.clip(BASELINE_WEIGHT + tilt, 0.0, 1.0))
    other = (1.0 - gld) / 3.0
    return pd.Series(
        {"ACWI": other, "AGG": other, "GLD": gld, "BSV": other}
    ).reindex(UNIVERSE)


## 5. Load prices and build weekly weights history

In [8]:
prices = load_etf_prices(UNIVERSE, start="2008-01-01")
weekly_prices = prices.resample("W-FRI").last()
weekly_returns = weekly_prices.pct_change()

DATA_END = weekly_prices.index.max()
print(f"Weekly price index: {weekly_prices.index.min().date()} to {DATA_END.date()} ({len(weekly_prices)} weeks)")
print(f"TRAIN_END:  {TRAIN_END.date()}")
print(f"VAL_END:    {VAL_END.date()}")


Weekly price index: 2008-01-04 to 2026-05-29 (961 weeks)
TRAIN_END:  2024-12-31
VAL_END:    2025-12-31


In [9]:
friday_grid = pd.date_range("2008-01-04", DATA_END, freq="W-FRI")
weights_history = pd.DataFrame(
    [get_weights(d) for d in friday_grid],
    index=friday_grid,
).reindex(columns=UNIVERSE)

print(f"Weights history: {len(weights_history)} weekly observations")
print(f"GLD weight range: {weights_history['GLD'].min():.2%} to {weights_history['GLD'].max():.2%} (baseline 25%)")

weights_history.tail(8)


Weights history: 961 weekly observations
GLD weight range: 10.04% to 39.89% (baseline 25%)


,ACWI,AGG,GLD,BSV
2026-04-10,0.2616,0.2616,0.2152,0.2616
2026-04-17,0.2616,0.2616,0.2152,0.2616
2026-04-24,0.2616,0.2616,0.2152,0.2616
2026-05-01,0.2611,0.2611,0.2168,0.2611
2026-05-08,0.2611,0.2611,0.2168,0.2611
2026-05-15,0.2611,0.2611,0.2168,0.2611
2026-05-22,0.2611,0.2611,0.2168,0.2611
2026-05-29,0.2611,0.2611,0.2168,0.2611


## 6. KPI tables on TRAIN, 2025 validation, and 2026 YTD

In [10]:
def lagged_portfolio_returns(weights: pd.DataFrame, returns: pd.DataFrame) -> pd.Series:
    aligned = weights.reindex(returns.index, method="ffill")
    return (aligned.shift() * returns).sum(axis=1, skipna=False).dropna()


strat = lagged_portfolio_returns(weights_history, weekly_returns)
bench = lagged_portfolio_returns(
    pd.DataFrame(0.25, index=friday_grid, columns=UNIVERSE),
    weekly_returns,
)

segments = {
    "TRAIN (through 2024-12-31)": (None, TRAIN_END),
    "VAL 2025": (TRAIN_END, VAL_END),
    "2026 YTD": (VAL_END, None),
}

records = []
for label, (lo, hi) in segments.items():
    mask = pd.Series(True, index=strat.index)
    if lo is not None:
        mask &= strat.index > lo
    if hi is not None:
        mask &= strat.index <= hi
    s = strat[mask]
    b = bench.reindex(s.index)
    records.append({"segment": label, "strategy": "Fundamental", **compute_kpis(s, b)})
    records.append({"segment": label, "strategy": "Equal-weight", **compute_kpis(b)})

kpis_df = pd.DataFrame(records).set_index(["segment", "strategy"])
kpis_df


n_periods  total_return   CAGR  \
segment                    strategy                                       
TRAIN (through 2024-12-31) Fundamental         874        1.2798 0.0503   
                           Equal-weight        874        1.2643 0.0498   
VAL 2025                   Fundamental          52        0.2668 0.2668   
                           Equal-weight         52        0.2530 0.2530   
2026 YTD                   Fundamental          22        0.0215 0.0516   
                           Equal-weight         22        0.0333 0.0806   

                                         ann_return  ann_vol  Sharpe  Sortino  \
segment                    strategy                                             
TRAIN (through 2024-12-31) Fundamental       0.0521   0.0777  0.6705   0.9741   
                           Equal-weight      0.0515   0.0749  0.6872   1.0018   
VAL 2025                   Fundamental       0.2388   0.0605  3.9499   9.0232   
                           Equal-weight      0.2274   0.0540  4.2086   9.0871   
2026 YTD                   Fundamental       0.0554   0.1025     NaN      NaN   
                           Equal-weight      0.0828   0.1042     NaN      NaN   

                                         max_drawdown  calmar  hit_rate  \
segment                    strategy                                       
TRAIN (through 2024-12-31) Fundamental        -0.1998  0.2515    0.5492   
                           Equal-weight       -0.1919  0.2597    0.5538   
VAL 2025                   Fundamental        -0.0212 12.5585    0.7115   
                           Equal-weight       -0.0217 11.6807    0.7500   
2026 YTD                   Fundamental        -0.0708  0.7284    0.5909   
                           Equal-weight       -0.0670  1.2021    0.5909   

                                         best_period  worst_period  \
segment                    strategy                                  
TRAIN (through 2024-12-31) Fundamental        0.0713       -0.0911   
                           Equal-weight       0.0679       -0.0751   
VAL 2025                   Fundamental        0.0273       -0.0212   
                           Equal-weight       0.0209       -0.0217   
2026 YTD                   Fundamental        0.0224       -0.0370   
                           Equal-weight       0.0224       -0.0333   

                                         active_sharpe_vs_bench  \
segment                    strategy                               
TRAIN (through 2024-12-31) Fundamental                   0.0369   
                           Equal-weight                     NaN   
VAL 2025                   Fundamental                   0.7728   
                           Equal-weight                     NaN   
2026 YTD                   Fundamental                      NaN   
                           Equal-weight                     NaN   

                                         excess_ann_return  
segment                    strategy                         
TRAIN (through 2024-12-31) Fundamental              0.0006  
                           Equal-weight                NaN  
VAL 2025                   Fundamental              0.0114  
                           Equal-weight                NaN  
2026 YTD                   Fundamental             -0.0274  
                           Equal-weight                NaN

In [11]:
# Persist for orchestration: history of weekly weights + KPI snapshot
weights_history.to_csv(RESULTS_DIR / "weekly_weights_history.csv")
kpis_df.to_csv(RESULTS_DIR / "kpis_segments.csv")
print(f"wrote {RESULTS_DIR / 'weekly_weights_history.csv'}")
print(f"wrote {RESULTS_DIR / 'kpis_segments.csv'}")


wrote c:\Users\sacha\Downloads\algo-trading-group\jan-fundamental\results\weekly_weights_history.csv
wrote c:\Users\sacha\Downloads\algo-trading-group\jan-fundamental\results\kpis_segments.csv


## 7. Submission for the upcoming Friday

In [12]:
_today = pd.Timestamp.today().normalize()
_days_to_fri = (4 - _today.weekday()) % 7
TARGET_DATE = (_today + pd.Timedelta(days=_days_to_fri)).strftime("%Y-%m-%d")
target = pd.Timestamp(TARGET_DATE)
assert target.day_name() == "Friday", f"{TARGET_DATE} is a {target.day_name()}, not a Friday"

weights_target = get_weights(target)
print(f"Weights for {TARGET_DATE} ({target.day_name()}):")
print(weights_target.map(lambda x: f"{x:.4f}").to_string())
print(f"sum = {float(weights_target.sum()):.6f}")


Weights for 2026-05-29 (Friday):
ACWI    0.2611
AGG     0.2611
GLD     0.2168
BSV     0.2611
sum = 1.000000


In [13]:
out_path = write_submission_csv(
    weights_target,
    target,
    TEAM_ID,
    SUBMISSIONS_DIR,
    vertical_tag="fundamental",
)
print(f"wrote: {out_path}")
print(out_path.read_text())


wrote: c:\Users\sacha\Downloads\algo-trading-group\jan-fundamental\submissions\TeamXX_fundamental_2026-05-29.csv
week,team_id,acwi,agg,gld,bsv
2026-05-29,TeamXX,26.10,26.11,21.68,26.11



## 8. Notes / next steps

- This notebook is the parity artifact the orchestrator (Rayane) will read. It exposes:
  - `get_weights(as_of_date)`: canonical signature, Series in [0,1] summing to 1.
  - `RESULTS_DIR / "weekly_weights_history.csv"`: full Friday-grid history.
  - `SUBMISSIONS_DIR / "TeamXX_fundamental_<YYYY-MM-DD>.csv"`: weekly draft.
- Strategy logic is intentionally unchanged from `src/fundamental.py`. If the production module changes, mirror the change in section 4 here.
- `TEAM_ID = "TeamXX"` is a placeholder. Update in the canonical constants block (section 1) before producing a final submission. The combined-final filename (no `vertical_tag`) is produced by the orchestrator.
- The canonical blocks are delimited with `# === BEGIN canonical:<name> ===` markers. Do not edit them in this notebook; edit `shared/canonical.py` and re-sync.
